# GDP Estimates and ROC Arrays for `max_grad_norm=1.0`

This notebook focuses on seeds `0-4` under `exp_data/max_grad_norm/1.0`.
It loads `emp_mu_gdp.npy`, `mia_fpr.npy`, and `mia_tpr.npy` for each available run,
prints missing seed/run combinations, prints each seed independently, and then
reconstructs a stronger order-2 RDP baseline from `losses_out.npy` and
`losses_in.npy`.

The final section mirrors the repo's finite-sample correction but keeps the
estimate RDP-native: for each threshold it builds Clopper-Pearson intervals for
`FPR` and `FNR`, converts those to an interval for `TPR`, and then lower-bounds
`D_2(Bern(TPR) || Bern(FPR))` term-by-term using conservative interval endpoints.


In [6]:
import ast
import math
import struct
import sys
from array import array
from pathlib import Path

results_root = Path("exp_data/max_grad_norm/1.0")
seeds = [0, 1, 2, 3, 4]


def split_run_name(run_name: str):
    if "_eps" not in run_name:
        return run_name, "", ""

    prefix, epsilon = run_name.rsplit("_eps", 1)
    if "_" not in prefix:
        return prefix, "", epsilon

    dataset, model = prefix.rsplit("_", 1)
    return dataset, model, epsilon


def epsilon_sort_key(epsilon: str):
    try:
        return (0, float(epsilon))
    except ValueError:
        return (1, epsilon)


def discover_run_names(root: Path, seed_ids: list[int]) -> list[str]:
    run_names = set()
    for seed in seed_ids:
        seed_dir = root / f"seed{seed}"
        if not seed_dir.exists():
            continue
        for run_dir in seed_dir.iterdir():
            if run_dir.is_dir():
                run_names.add(run_dir.name)

    return sorted(
        run_names,
        key=lambda name: (
            split_run_name(name)[0],
            split_run_name(name)[1],
            epsilon_sort_key(split_run_name(name)[2]),
            name,
        ),
    )


def load_npy_float64(path: Path):
    with path.open("rb") as handle:
        if handle.read(6) != b"\x93NUMPY":
            raise ValueError(f"{path} is not a valid .npy file")

        major, minor = struct.unpack("BB", handle.read(2))
        if major == 1:
            header_len = struct.unpack("<H", handle.read(2))[0]
        elif major in (2, 3):
            header_len = struct.unpack("<I", handle.read(4))[0]
        else:
            raise ValueError(f"Unsupported .npy version {major}.{minor} in {path}")

        header = ast.literal_eval(handle.read(header_len).decode("latin1"))
        descr = header["descr"]
        if descr != "<f8":
            raise ValueError(f"Expected float64 data in {path}, found {descr}")

        shape = header["shape"]
        count = 1 if shape == () else math.prod(shape)
        values = array("d")
        values.frombytes(handle.read(count * 8))

    if sys.byteorder != "little":
        values.byteswap()

    return values[0] if count == 1 else values


run_names = discover_run_names(results_root, seeds)
run_names


['cifar10_half_cnn_eps2.0',
 'cifar10_half_cnn_eps4.38',
 'cifar10_half_cnn_eps6.57',
 'cifar10_half_cnn_eps10.0',
 'cifar10_half_cnn_eps17.85',
 'mnist_half_cnn_eps2.0',
 'mnist_half_cnn_eps4.38',
 'mnist_half_cnn_eps6.57',
 'mnist_half_cnn_eps10.0',
 'mnist_half_cnn_eps17.85']

In [7]:
results = {}
missing = []

for run_name in run_names:
    per_seed = {}
    for seed in seeds:
        run_dir = results_root / f"seed{seed}" / run_name
        mu_path = run_dir / "emp_mu_gdp.npy"
        fpr_path = run_dir / "mia_fpr.npy"
        tpr_path = run_dir / "mia_tpr.npy"

        if mu_path.exists() and fpr_path.exists() and tpr_path.exists():
            per_seed[seed] = {
                "emp_mu_gdp": load_npy_float64(mu_path),
                "mia_fpr": load_npy_float64(fpr_path),
                "mia_tpr": load_npy_float64(tpr_path),
            }
        else:
            missing.append((run_name, seed))

    results[run_name] = per_seed

print("Missing seed/run combinations:")
if missing:
    for run_name, seed in missing:
        print(f"  {run_name} | seed {seed}")
else:
    print("  none")


Missing seed/run combinations:
  none


In [8]:
for run_name in run_names:
    print(f"=== {run_name} ===")
    available_seeds = sorted(results[run_name])

    if not available_seeds:
        print("No completed seeds found.")
        print()
        continue

    for seed in available_seeds:
        seed_result = results[run_name][seed]
        print(f"seed {seed}")
        print("  mu_gdp  =", seed_result["emp_mu_gdp"])
        print("  mia_fpr =", seed_result["mia_fpr"])
        print("  mia_tpr =", seed_result["mia_tpr"])
        print()


=== cifar10_half_cnn_eps2.0 ===
seed 0
  mu_gdp  = 0.23889941888602872
  mia_fpr = array('d', [1.0, 0.99, 0.99, 0.98, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.92, 0.92, 0.92, 0.92, 0.91, 0.9, 0.9, 0.9, 0.9, 0.89, 0.89, 0.88, 0.88, 0.87, 0.87, 0.86, 0.86, 0.85, 0.84, 0.84, 0.83, 0.82, 0.81, 0.81, 0.8, 0.79, 0.78, 0.78, 0.77, 0.76, 0.75, 0.75, 0.74, 0.73, 0.72, 0.72, 0.72, 0.72, 0.71, 0.71, 0.7, 0.7, 0.69, 0.68, 0.68, 0.67, 0.66, 0.66, 0.66, 0.65, 0.65, 0.64, 0.64, 0.63, 0.62, 0.62, 0.61, 0.61, 0.6, 0.6, 0.59, 0.58, 0.58, 0.58, 0.57, 0.56, 0.55, 0.54, 0.53, 0.52, 0.51, 0.51, 0.5, 0.5, 0.49, 0.49, 0.49, 0.48, 0.47, 0.47, 0.46, 0.45, 0.44, 0.43, 0.42, 0.42, 0.41, 0.4, 0.39, 0.39, 0.39, 0.38, 0.38, 0.38, 0.37, 0.37, 0.36, 0.35, 0.34, 0.33, 0.32, 0.32, 0.32, 0.31, 0.31, 0.31, 0.31, 0.3, 0.3, 0.3, 0.29, 0.28, 0.27, 0.27, 0.26, 0.26, 0.26, 0.25, 0.24, 0.24, 0.24, 0.23, 0.22, 0.22, 0.21, 0.2, 0.2, 0.19, 0.19, 0.19, 0.18, 0.18, 0.18, 0.17, 0.16, 0.16, 0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.15,

In [10]:
from statistics import NormalDist

normal = NormalDist()
audit_alpha = 0.0125
corrected_rdp_alpha = 2.0
corrected_bern_rdp_results = {}


def clipped_inv_cdf(p: float, clip: float = 1e-12) -> float:
    p = min(max(float(p), clip), 1.0 - clip)
    return normal.inv_cdf(p)


def load_losses_for_seed(run_name: str, seed: int):
    run_dir = results_root / f"seed{seed}" / run_name
    losses_out = load_npy_float64(run_dir / "losses_out.npy")
    losses_in = load_npy_float64(run_dir / "losses_in.npy")
    return losses_out, losses_in


def binom_cdf_leq(k: int, n: int, p: float) -> float:
    if k < 0:
        return 0.0
    if k >= n:
        return 1.0
    if p <= 0.0:
        return 1.0
    if p >= 1.0:
        return 0.0

    total = 0.0
    q = 1.0 - p
    for i in range(k + 1):
        total += math.comb(n, i) * (p ** i) * (q ** (n - i))
    return min(max(total, 0.0), 1.0)


def clopper_pearson_upper(successes: int, trials: int, alpha: float) -> float:
    if trials <= 0:
        return 1.0
    if successes >= trials:
        return 1.0

    lo = 0.0
    hi = 1.0
    for _ in range(80):
        mid = (lo + hi) / 2.0
        if binom_cdf_leq(successes, trials, mid) > alpha:
            lo = mid
        else:
            hi = mid
    return hi


def clopper_pearson_lower(successes: int, trials: int, alpha: float) -> float:
    if trials <= 0 or successes <= 0:
        return 0.0

    lo = 0.0
    hi = 1.0
    target = 1.0 - alpha
    for _ in range(80):
        mid = (lo + hi) / 2.0
        if binom_cdf_leq(successes - 1, trials, mid) > target:
            lo = mid
        else:
            hi = mid
    return lo


def renyi_alpha2_lower_bound_from_intervals(p_lower: float, p_upper: float, q_lower: float, q_upper: float) -> float:
    if corrected_rdp_alpha != 2.0:
        raise ValueError("This notebook cell is written for order-2 RDP")

    term1 = 0.0 if p_lower <= 0.0 else (p_lower * p_lower) / q_upper
    term2_num = 1.0 - p_upper
    term2 = 0.0 if term2_num <= 0.0 else (term2_num * term2_num) / (1.0 - q_lower)
    value = math.log(term1 + term2)
    if math.isnan(value) or value < 0.0:
        return 0.0
    return value


def estimate_corrected_bern_rdp_from_losses(losses_out, losses_in, alpha: float):
    thresholds = sorted({float(value) for value in losses_out} | {float(value) for value in losses_in})
    n_out = len(losses_out)
    n_in = len(losses_in)
    best = None

    for tau in thresholds:
        fp = sum(1 for loss in losses_out if float(loss) >= tau)
        fn = sum(1 for loss in losses_in if float(loss) < tau)
        tp = n_in - fn
        tn = n_out - fp

        raw_fpr = fp / n_out if n_out else 0.0
        raw_tpr = tp / n_in if n_in else 0.0

        fpr_lower = clopper_pearson_lower(fp, n_out, alpha)
        fpr_upper = clopper_pearson_upper(fp, n_out, alpha)
        fnr_lower = clopper_pearson_lower(fn, n_in, alpha)
        fnr_upper = clopper_pearson_upper(fn, n_in, alpha)
        tpr_lower = max(0.0, 1.0 - fnr_upper)
        tpr_upper = max(0.0, 1.0 - fnr_lower)

        mu_lower = clipped_inv_cdf(tpr_lower) - clipped_inv_cdf(fpr_upper)
        if math.isnan(mu_lower) or mu_lower < 0.0:
            mu_lower = 0.0

        renyi_lower = renyi_alpha2_lower_bound_from_intervals(
            p_lower=tpr_lower,
            p_upper=tpr_upper,
            q_lower=fpr_lower,
            q_upper=fpr_upper,
        )

        candidate = {
            "rdp_tau": float(tau),
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "tn": tn,
            "n_out": n_out,
            "n_in": n_in,
            "raw_fpr": float(raw_fpr),
            "raw_tpr": float(raw_tpr),
            "fpr_lower": float(fpr_lower),
            "fpr_upper": float(fpr_upper),
            "fnr_lower": float(fnr_lower),
            "fnr_upper": float(fnr_upper),
            "tpr_lower": float(tpr_lower),
            "tpr_upper": float(tpr_upper),
            "mu_lower_at_rdp_tau": float(mu_lower),
            "renyi_alpha2_lower_bern": float(renyi_lower),
        }

        if best is None or candidate["renyi_alpha2_lower_bern"] > best["renyi_alpha2_lower_bern"]:
            best = candidate

    return best


for run_name in run_names:
    available_seeds = sorted(results[run_name])
    per_seed = {}

    for seed in available_seeds:
        losses_out, losses_in = load_losses_for_seed(run_name, seed)
        per_seed[seed] = estimate_corrected_bern_rdp_from_losses(
            losses_out,
            losses_in,
            alpha=audit_alpha,
        )

    corrected_bern_rdp_results[run_name] = per_seed


In [ ]:
cp_confidence_level = 1.0 - 4.0 * audit_alpha

for run_name in run_names:
    print(
        f"=== {run_name} | corrected Bernoulli RDP lower bound (alpha=2, CP level={cp_confidence_level:.2f}) ==="
    )
    available_seeds = sorted(corrected_bern_rdp_results[run_name])

    if not available_seeds:
        print("No completed seeds found.")
        print()
        continue

    for seed in available_seeds:
        summary = corrected_bern_rdp_results[run_name][seed]
        print(f"seed {seed}")
        print("  rdp_tau                 =", summary["rdp_tau"])
        print("  counts_fp_fn_tp_tn      =", (summary["fp"], summary["fn"], summary["tp"], summary["tn"]))
        print("  raw_fpr                 =", summary["raw_fpr"])
        print("  raw_tpr                 =", summary["raw_tpr"])
        print("  fpr_ci                  =", (summary["fpr_lower"], summary["fpr_upper"]))
        print("  fnr_ci                  =", (summary["fnr_lower"], summary["fnr_upper"]))
        print("  tpr_ci                  =", (summary["tpr_lower"], summary["tpr_upper"]))
        print("  mu_lower_at_rdp_tau     =", summary["mu_lower_at_rdp_tau"])
        print("  renyi_alpha2_lower_bern =", summary["renyi_alpha2_lower_bern"])
        print()


=== cifar10_half_cnn_eps2.0 | corrected Bernoulli RDP lower bound (alpha=2, CP level=0.90) ===
seed 0
  rdp_tau                 = -10.482940673828125
  counts_fp_fn_tp_tn      = (100, 0, 100, 0)
  raw_fpr                 = 1.0
  raw_tpr                 = 1.0
  fpr_ci                  = (0.97048695039296, 1.0)
  fnr_ci                  = (0.0, 0.02951304960703999)
  tpr_ci                  = (0.97048695039296, 1.0)
  mu_lower_at_rdp_tau     = 0.0
  renyi_alpha2_lower_bern = 0.0

seed 1
  rdp_tau                 = -9.40695858001709
  counts_fp_fn_tp_tn      = (100, 0, 100, 0)
  raw_fpr                 = 1.0
  raw_tpr                 = 1.0
  fpr_ci                  = (0.97048695039296, 1.0)
  fnr_ci                  = (0.0, 0.02951304960703999)
  tpr_ci                  = (0.97048695039296, 1.0)
  mu_lower_at_rdp_tau     = 0.0
  renyi_alpha2_lower_bern = 0.0

seed 2
  rdp_tau                 = -9.815881729125977
  counts_fp_fn_tp_tn      = (100, 0, 100, 0)
  raw_fpr                 = 1.0
